# 5 · End-to-end check: the actual deployed server, over the real protocol

Notebooks 2-4 checked pieces of the system in-process. This is the final check: start
`mcp_server.py` **exactly as it actually runs** (same file, same startup path,
`load_all_indexes()` and all) as a real subprocess, and talk to it over genuine MCP
protocol (JSON-RPC over stdio) using the official client SDK — the same way Claude
Desktop or Claude Code would.

Requires notebook 1 to have been run — `mcp_server.py`'s own startup will raise the same
clear "run notebook 1 first" error this notebook would otherwise reproduce, since it's
the literal same `load_all_indexes()` call underneath.

In [ ]:
%%capture
!pip install -q -r requirements.txt


In [ ]:
# Get the project files (config.py, portfolio.py, data/) if they aren't
# already here -- lets this notebook be opened and run on its own in Colab.
import os, subprocess, sys

if not os.path.exists("portfolio.py"):
    if os.path.exists("../portfolio.py"):
        os.chdir("..")
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/hossamhamdy333/AI_Portfolio.git", "repo"],
            check=True,
        )
        os.chdir("repo/Codebase_Insight_Agent")

sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())


In [ ]:
from getpass import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Google API key (Gemini): ")

# QDRANT_URL/QDRANT_API_KEY make the index PERSISTENT (Qdrant Cloud, free
# tier is enough) instead of rebuilt from scratch every time. This matters
# specifically because this notebook runs in a fresh Colab VM each time,
# separate from wherever mcp_server.py/web_app.py actually run - without a
# real, shared QDRANT_URL, this notebook's work never reaches those
# services at all. Get a free instance at https://cloud.qdrant.io
if not os.environ.get("QDRANT_URL"):
    os.environ["QDRANT_URL"] = input("Qdrant Cloud URL (blank = local in-memory, no persistence): ")
if os.environ["QDRANT_URL"] and not os.environ.get("QDRANT_API_KEY"):
    os.environ["QDRANT_API_KEY"] = getpass("Qdrant API key: ")


In [ ]:
import pandas as pd

eval_questions = pd.read_csv("data/eval_set_starter.csv")
print(f"{len(eval_questions)} regression questions loaded")
eval_questions[["question", "expected_projects"]]


In [ ]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command=sys.executable,
    args=["mcp_server.py"],
    env=dict(os.environ),
    cwd=os.getcwd(),
)

async def run_end_to_end_check():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await session.list_tools()
            print("Tools exposed over MCP:", [t.name for t in tools.tools])

            # Two real regression questions, asked over the actual protocol -
            # not a synthetic ping, the same questions notebooks 2-4 already
            # validated, now proving the deployed server gives the same
            # correct answer over MCP that it gave in-process.
            for _, row in eval_questions.head(2).iterrows():
                print(f"\n--- {row['question']} ---")
                result = await session.call_tool("ask_portfolio", {"question": row["question"]})
                answer = result.content[0].text
                print(answer[:300])
                expected = [p.strip() for p in row["expected_projects"].split(",")]
                mentions_expected = all(p in answer for p in expected)
                print(f"[mentions {expected}: {mentions_expected}]")

await run_end_to_end_check()


If this cell raised a `RuntimeError` about a missing index, that's
`mcp_server.py` itself telling you notebook 1 hasn't been run against a real
`QDRANT_URL` yet — go do that first, then come back to this cell.